<a href="https://colab.research.google.com/github/mynameisBhagya/ids-cicids2017/blob/main/notebooks/CICIDS2017_Intrusion_Detection_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Machine Learning-Based Network Intrusion Detection and Multi-Class Attack Classification

## Project Overview

This project develops a machine learning-based network intrusion detection system using the CIC-IDS2017 dataset.

The project aims to identify malicious network traffic and classify different types of cyberattacks using machine learning algorithms.

## Project Objectives

1. Perform exploratory data analysis on the CIC-IDS2017 dataset.
2. Clean and preprocess network traffic data.
3. Develop a binary intrusion detection model.
4. Develop a multiclass attack classification model.
5. Compare Logistic Regression, Decision Tree, Random Forest, XGBoost and SVM.
6. Evaluate model performance using appropriate classification metrics.
7. Investigate data leakage and class imbalance.
8. Apply SHAP for model explainability.
9. Present the experimental results through visualisations.

## Dataset

CIC-IDS2017 — Canadian Institute for Cybersecurity Intrusion Detection Dataset.

Dataset source:
https://www.kaggle.com/datasets/dhoogla/cicids2017

## Development Environment

- Python
- Google Colab
- Pandas and NumPy
- Scikit-learn
- XGBoost
- Matplotlib and Seaborn
- SHAP


 **Import the libraries**

imports the Python libraries that we will use throughout the project. We set a fixed random seed to make experiments more reproducible.

In [1]:
# Import pandas for handling tabular datasets.
import pandas as pd

# Import NumPy for numerical calculations.
import numpy as np

# Import os for basic file and path operations.
import os

# Import time for measuring model training duration.
import time

# Import matplotlib for creating graphs.
import matplotlib.pyplot as plt

# Import seaborn for statistical visualisations.
import seaborn as sns

# Set a fixed random seed for repeatable experiments.
RANDOM_STATE = 42

# Set a consistent plotting style.
sns.set_theme(style="whitegrid")

# Display a confirmation message.
print("Project libraries imported successfully!")

Project libraries imported successfully!


**Install machine learning packages:**

These packages provide tools for XGBoost, handling imbalanced classes and explaining model predictions. Scikit-learn, Pandas, NumPy and plotting libraries are already available in most Colab environments.

In [2]:
# Install XGBoost for gradient-boosted tree classification.
%pip -q install xgboost

# Install imbalanced-learn for class imbalance experiments.
%pip -q install imbalanced-learn

# Install SHAP for explaining machine learning predictions.
%pip -q install shap

# Display a confirmation message.
print("Additional project packages installed!")

Additional project packages installed!


**Uploading eight Parquet files:**




In [4]:
# Import the Colab file-upload function.
from google.colab import files

# Open the file-selection window on your computer.
uploaded = files.upload()

# Find all uploaded files ending in .parquet.
parquet_files = [
    filename
    for filename in uploaded.keys()
    if filename.lower().endswith(".parquet")
]

# Display how many Parquet files were uploaded.
print("Number of Parquet files:", len(parquet_files))

# Display the uploaded filenames.
for filename in parquet_files:
    print(filename)

Saving WebAttacks-Thursday-no-metadata.parquet to WebAttacks-Thursday-no-metadata.parquet
Saving DoS-Wednesday-no-metadata.parquet to DoS-Wednesday-no-metadata.parquet
Saving Infiltration-Thursday-no-metadata.parquet to Infiltration-Thursday-no-metadata.parquet
Saving Portscan-Friday-no-metadata.parquet to Portscan-Friday-no-metadata.parquet
Saving Bruteforce-Tuesday-no-metadata.parquet to Bruteforce-Tuesday-no-metadata.parquet
Saving DDoS-Friday-no-metadata.parquet to DDoS-Friday-no-metadata.parquet
Saving Benign-Monday-no-metadata.parquet to Benign-Monday-no-metadata.parquet
Saving Botnet-Friday-no-metadata.parquet to Botnet-Friday-no-metadata.parquet
Number of Parquet files: 8
WebAttacks-Thursday-no-metadata.parquet
DoS-Wednesday-no-metadata.parquet
Infiltration-Thursday-no-metadata.parquet
Portscan-Friday-no-metadata.parquet
Bruteforce-Tuesday-no-metadata.parquet
DDoS-Friday-no-metadata.parquet
Benign-Monday-no-metadata.parquet
Botnet-Friday-no-metadata.parquet


**Checking uploaded file sizes:**

This confirms that the files have been uploaded and lets us see how large they are. It is useful because large datasets can consume Colab's available memory.

In [5]:
# Check the size of each uploaded file.
for filename in parquet_files:

    # Calculate the file size in megabytes.
    file_size = os.path.getsize(filename) / (1024 * 1024)

    # Display the filename and its size.
    print(filename, ":", round(file_size, 2), "MB")

WebAttacks-Thursday-no-metadata.parquet : 16.84 MB
DoS-Wednesday-no-metadata.parquet : 65.04 MB
Infiltration-Thursday-no-metadata.parquet : 22.07 MB
Portscan-Friday-no-metadata.parquet : 12.96 MB
Bruteforce-Tuesday-no-metadata.parquet : 44.0 MB
DDoS-Friday-no-metadata.parquet : 24.13 MB
Benign-Monday-no-metadata.parquet : 54.14 MB
Botnet-Friday-no-metadata.parquet : 18.94 MB


**Read and combine the eight Parquet files:**

This reads the eight files and combines their rows into one dataset called df. The following project sections will use this DataFrame.

This assumes the eight files have compatible columns. If Colab runs out of memory, stop and tell me rather than repeatedly executing the cell.

In [6]:
# Create an empty list to temporarily hold the eight DataFrames.
dataframes = []

# Read each Parquet file one at a time.
for filename in parquet_files:

    # Display the name of the file currently being read.
    print("Reading:", filename)

    # Load the current Parquet file into a DataFrame.
    temporary_df = pd.read_parquet(filename)

    # Add the DataFrame to the list.
    dataframes.append(temporary_df)

# Combine all eight DataFrames vertically into one dataset.
df = pd.concat(dataframes, ignore_index=True)

# Remove the temporary list to release some memory.
del dataframes

# Display the combined dataset dimensions.
print("Dataset successfully combined!")
print("Total rows:", df.shape[0])
print("Total columns:", df.shape[1])

Reading: WebAttacks-Thursday-no-metadata.parquet
Reading: DoS-Wednesday-no-metadata.parquet
Reading: Infiltration-Thursday-no-metadata.parquet
Reading: Portscan-Friday-no-metadata.parquet
Reading: Bruteforce-Tuesday-no-metadata.parquet
Reading: DDoS-Friday-no-metadata.parquet
Reading: Benign-Monday-no-metadata.parquet
Reading: Botnet-Friday-no-metadata.parquet
Dataset successfully combined!
Total rows: 2313810
Total columns: 78


**Previewing the combined dataset:**

gives an initial look at the dataset and its structure. We need to understand the columns before selecting the features for machine learning.

In [7]:
# Remove unnecessary spaces from the column names.
df.columns = df.columns.str.strip()

# Display the first five records.
display(df.head())

# Display the last five records.
display(df.tail())

# Display the number of rows and columns.
print("Dataset shape:", df.shape)

# Display all column names.
print("\nDataset columns:")
print(df.columns.tolist())

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,6,113095465,48,24,9668,10012,403,0,201.416672,203.548294,...,32,203985.500,5.758372e+05,1629110,379,13800000.0,4277541.0,16500000,6737603,Benign
1,6,113473706,68,40,11364,12718,403,0,167.117645,171.919418,...,32,178326.875,5.034269e+05,1424245,325,13800000.0,4229413.0,16500000,6945512,Benign
2,0,119945515,150,0,0,0,0,0,0.000000,0.000000,...,0,6909777.500,1.170000e+07,20400000,6,24400000.0,24300000.0,60100000,5702188,Benign
3,6,60261928,9,7,2330,4221,1093,0,258.888885,409.702148,...,20,0.000,0.000000e+00,0,0,0.0,0.0,0,0,Benign
4,17,269,2,2,102,322,51,51,51.000000,0.000000,...,32,0.000,0.000000e+00,0,0,0.0,0.0,0,0,Benign


,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
2313805,17,61452,4,2,180,354,45,45,45.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2313806,17,171,2,2,80,272,40,40,40.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2313807,17,222,2,2,90,354,45,45,45.0,0.0,...,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2313808,17,16842,1,1,48,48,48,48,48.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2313809,17,153,2,2,68,100,34,34,34.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign


Dataset shape: (2313810, 78)

Dataset columns:
['Protocol', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Fwd Packets Length Total', 'Bwd Packets Length Total', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 

**Identifying the target label:**

The label column contains the correct category for each network record. It is the answer that the classification models will learn to predict.

In [8]:
# Find columns whose name is exactly 'label', ignoring capitalisation.
label_candidates = [
    column for column in df.columns
    if column.lower() == "label"
]

# Display the possible target columns.
print("Label column candidates:", label_candidates)

Label column candidates: ['Label']


In [9]:
# Select the dataset's label column.
label_column = label_candidates[0]

# Display the different traffic categories and their counts.
print(df[label_column].value_counts(dropna=False))

Label
Benign                        1977318
DoS Hulk                       172846
DDoS                           128014
DoS GoldenEye                   10286
FTP-Patator                      5931
DoS slowloris                    5385
DoS Slowhttptest                 5228
SSH-Patator                      3219
PortScan                         1956
Web Attack � Brute Force         1470
Bot                              1437
Web Attack � XSS                  652
Infiltration                       36
Web Attack � Sql Injection         21
Heartbleed                         11
Name: count, dtype: int64
